# Dwarf galaxies of Andromeda

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Ellipse, Circle
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "andromeda_dark_dwarf_satellites_layer.gif"

FPS = 24
DURATION_SEC = 12
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#020510"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
DEEP_BLUE = "#0b63ff"
ORANGE = "#ff9a3c"
WHITE = "#f0f6ff"
PURPLE = "#b76cff"

rng = np.random.default_rng(136)

# =========================
# Scene geometry
# =========================

W, H = 16, 9
N_STARS = 420
N_BACKGROUND_GALAXIES = 80
N_SATELLITES = 18

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_position([0, 0, 1, 1])

PAD = 0.22

ax.set_xlim(PAD, W - PAD)
ax.set_ylim(PAD, H - PAD)
ax.set_aspect("equal")
ax.axis("off")

# =========================
# Star background
# =========================

star_x = rng.uniform(0, W, N_STARS)
star_y = rng.uniform(0, H, N_STARS)
star_s = rng.uniform(0.5, 8.0, N_STARS)
star_alpha = rng.uniform(0.12, 0.72, N_STARS)

ax.scatter(
    star_x,
    star_y,
    s=star_s,
    c=WHITE,
    alpha=star_alpha,
    linewidths=0,
    zorder=1
)

# Brighter foreground stars
for _ in range(18):
    sx = rng.uniform(0.5, W - 0.5)
    sy = rng.uniform(0.5, H - 0.5)
    ss = rng.uniform(12, 30)
    ax.scatter([sx], [sy], s=ss, c=WHITE, alpha=0.78, linewidths=0, zorder=2)
    ax.plot([sx - 0.08, sx + 0.08], [sy, sy], color=WHITE, alpha=0.28, linewidth=0.7, zorder=2)
    ax.plot([sx, sx], [sy - 0.08, sy + 0.08], color=WHITE, alpha=0.28, linewidth=0.7, zorder=2)

# =========================
# Background faint galaxies
# =========================

galaxy_palette = [
    "#f7d9a0", "#d7c7ff", "#ffb27a", "#dce7ff",
    "#b7c7ff", "#ffd0c2", "#ff8f70", "#b7e8ff"
]

for _ in range(N_BACKGROUND_GALAXIES):
    gx = rng.uniform(0.5, W - 0.5)
    gy = rng.uniform(0.5, H - 0.5)

    # avoid center a little; M31 will dominate
    if (gx - W / 2) ** 2 + (gy - H / 2) ** 2 < 4.5:
        continue

    width = rng.uniform(0.04, 0.18)
    height = width * rng.uniform(0.25, 0.85)
    angle = rng.uniform(0, 180)
    color = rng.choice(galaxy_palette)
    alpha = rng.uniform(0.12, 0.42)

    glow = Ellipse(
        (gx, gy),
        width * 2.2,
        height * 2.2,
        angle=angle,
        facecolor=color,
        edgecolor="none",
        alpha=alpha * 0.10,
        zorder=2
    )
    body = Ellipse(
        (gx, gy),
        width,
        height,
        angle=angle,
        facecolor=color,
        edgecolor="none",
        alpha=alpha,
        zorder=3
    )

    ax.add_patch(glow)
    ax.add_patch(body)

# =========================
# M31 schematic in center
# =========================

m31_x, m31_y = W / 2, H / 2
m31_angle = -13

# outer halo
for scale, alpha, color in [
    (4.2, 0.035, "#7da8ff"),
    (3.3, 0.055, "#89b6ff"),
    (2.4, 0.075, "#b7c7ff"),
]:
    halo = Ellipse(
        (m31_x, m31_y),
        width=scale,
        height=scale * 0.33,
        angle=m31_angle,
        facecolor=color,
        edgecolor="none",
        alpha=alpha,
        zorder=5
    )
    ax.add_patch(halo)

# disk layers
for scale, alpha, color in [
    (2.8, 0.22, "#6f91d8"),
    (2.2, 0.26, "#9eb5e8"),
    (1.5, 0.32, "#d7d3c7"),
    (0.75, 0.65, "#ffd9a4"),
]:
    disk = Ellipse(
        (m31_x, m31_y),
        width=scale,
        height=scale * 0.30,
        angle=m31_angle,
        facecolor=color,
        edgecolor="none",
        alpha=alpha,
        zorder=7
    )
    ax.add_patch(disk)

# nucleus glow
nucleus_glow = Circle(
    (m31_x, m31_y),
    radius=0.33,
    facecolor="#ffe7b7",
    edgecolor="none",
    alpha=0.24,
    zorder=8
)
nucleus = Circle(
    (m31_x, m31_y),
    radius=0.13,
    facecolor="#fff1d0",
    edgecolor="none",
    alpha=0.88,
    zorder=9
)
ax.add_patch(nucleus_glow)
ax.add_patch(nucleus)

# a few dust lanes / spiral-like strokes as ellipses
for offset, scale, alpha in [
    (-0.10, 2.25, 0.18),
    (0.08, 1.90, 0.14),
    (0.22, 1.45, 0.12),
]:
    lane = Ellipse(
        (m31_x + offset, m31_y + offset * 0.1),
        width=scale,
        height=scale * 0.22,
        angle=m31_angle,
        facecolor="none",
        edgecolor="#1b2438",
        linewidth=1.3,
        alpha=alpha,
        zorder=10
    )
    ax.add_patch(lane)

# =========================
# Satellite positions: chaotic field distribution
# =========================

def roman(n):
    numerals = [
        (1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
        (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
        (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")
    ]
    out = ""
    for value, symbol in numerals:
        while n >= value:
            out += symbol
            n -= value
    return out

# Selected Roman numbers, randomized, with XXXVI forced
roman_numbers = [13, 14, 16, 17, 18, 19, 20, 21, 23, 24, 25, 27, 28, 29, 30, 31, 32, 36]
rng.shuffle(roman_numbers)

# force Andromeda XXXVI to be present and visually distinct
if 36 not in roman_numbers:
    roman_numbers[-1] = 36

target_positions = []

attempts = 0
while len(target_positions) < N_SATELLITES and attempts < 5000:
    attempts += 1

    # Chaotic distribution: broad rectangle, but avoid exact M31 center
    x = rng.uniform(0.9, W - 0.9)
    y = rng.uniform(0.85, H - 0.9)

    dx = x - m31_x
    dy = y - m31_y

    dist2 = dx * dx + dy * dy

    # avoid hiding inside M31 disk, but allow some inner satellites
    if dist2 < 1.15:
        continue

    # keep labels and rings apart
    if all((x - px) ** 2 + (y - py) ** 2 > 0.70 for px, py, _ in target_positions):
        idx = len(target_positions)
        target_positions.append((x, y, roman_numbers[idx]))

# Make XXXVI not too close to edge and noticeable
xxxvi_idx = next(i for i, (_, _, n) in enumerate(target_positions) if n == 36)
target_positions[xxxvi_idx] = (6.6, 1.75, 36)

# =========================
# Satellite layer artists
# =========================

satellite_artists = []
ring_artists = []
label_artists = []
connector_artists = []

for i, (x, y, num) in enumerate(target_positions, start=1):
    is_xxxvi = num == 36

    size = rng.uniform(0.070, 0.145)
    ratio = rng.uniform(0.70, 1.20)
    angle = rng.uniform(0, 180)

    color = CYAN if not is_xxxvi else ORANGE
    glow_color = DEEP_BLUE if not is_xxxvi else ORANGE

    glow = Ellipse(
        (x, y),
        size * 6.0,
        size * 6.0 * ratio,
        angle=angle,
        facecolor=glow_color,
        edgecolor="none",
        alpha=0.0,
        zorder=20
    )

    # dark dwarf satellite: faint diffuse smudge
    body = Ellipse(
        (x, y),
        size,
        size * ratio,
        angle=angle,
        facecolor=color,
        edgecolor=WHITE,
        linewidth=0.3,
        alpha=0.0,
        zorder=21
    )

    core = Circle(
        (x, y),
        radius=size * 0.18,
        facecolor=WHITE,
        edgecolor="none",
        alpha=0.0,
        zorder=22
    )

    ring1 = Circle(
        (x, y),
        radius=size * 2.0,
        edgecolor=color,
        facecolor="none",
        linewidth=1.1,
        alpha=0.0,
        zorder=23
    )

    ring2 = Circle(
        (x, y),
        radius=size * 3.0,
        edgecolor=glow_color,
        facecolor="none",
        linewidth=0.8,
        alpha=0.0,
        zorder=23
    )

    # label offsets based on quadrant, randomized slightly
    sx = 1 if x < m31_x else -1
    sy = 1 if y < m31_y else -1

    dx = sx * rng.uniform(0.35, 0.62)
    dy = sy * rng.uniform(0.20, 0.46)

    if is_xxxvi:
        label_text = "Andromeda XXXVI"
        dx, dy = 0.42, 0.34
        ha = "left"
        fontsize = 10.0
    else:
        label_text = roman(num)
        ha = "left" if dx > 0 else "right"
        fontsize = 9.0

    label = ax.text(
        x + dx,
        y + dy,
        label_text,
        color=color,
        fontsize=fontsize,
        fontweight="bold",
        ha=ha,
        va="center",
        alpha=0.0,
        zorder=24
    )

    connector, = ax.plot(
        [x, x + dx * 0.72],
        [y, y + dy * 0.72],
        color=color,
        linewidth=0.65,
        alpha=0.0,
        zorder=23
    )

    for artist in [glow, body, core, ring1, ring2]:
        ax.add_patch(artist)

    satellite_artists.append((glow, body, core, is_xxxvi, size))
    ring_artists.append((ring1, ring2, size))
    label_artists.append(label)
    connector_artists.append(connector)

# =========================
# HUD overlay
# =========================

title = ax.text(
    0.5,
    8.55,
    "DARK DWARF SATELLITES OF ANDROMEDA (M31)",
    color=TEXT,
    fontsize=18,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=40
)

subtitle = ax.text(
    0.5,
    8.22,
    "satellite layer OFF",
    color=MUTED,
    fontsize=12,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=40
)

status_box = ax.text(
    15.45,
    8.45,
    "DWARF LAYER: OFF",
    color=MUTED,
    fontsize=11,
    fontweight="bold",
    ha="right",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358",
        alpha=0.85
    ),
    zorder=40
)

counter = ax.text(
    15.45,
    0.42,
    "0 satellites highlighted",
    color=MUTED,
    fontsize=10,
    ha="right",
    va="center",
    zorder=40
)

m31_label = ax.text(
    m31_x + 0.70,
    m31_y - 0.46,
    "M31",
    color=TEXT,
    fontsize=11,
    fontweight="bold",
    alpha=0.75,
    zorder=35
)

# subtle frame
frame_lines = []
frame_style = dict(color="#2b4358", linewidth=1.0, alpha=0.45, zorder=35)
frame_lines.append(ax.plot([0.35, 3.0], [8.75, 8.75], **frame_style)[0])
frame_lines.append(ax.plot([0.35, 0.35], [8.75, 7.95], **frame_style)[0])
frame_lines.append(ax.plot([13.0, 15.65], [8.75, 8.75], **frame_style)[0])
frame_lines.append(ax.plot([15.65, 15.65], [8.75, 7.95], **frame_style)[0])
frame_lines.append(ax.plot([0.35, 2.1], [0.25, 0.25], **frame_style)[0])
frame_lines.append(ax.plot([0.35, 0.35], [0.25, 0.95], **frame_style)[0])
frame_lines.append(ax.plot([13.9, 15.65], [0.25, 0.25], **frame_style)[0])
frame_lines.append(ax.plot([15.65, 15.65], [0.25, 0.95], **frame_style)[0])

# =========================
# Animation helpers
# =========================

def smoothstep(edge0, edge1, x):
    if edge0 == edge1:
        return 1.0 if x >= edge1 else 0.0
    t = np.clip((x - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

def layer_alpha(t):
    """
    Smooth loop:
    0.00-0.17: layer off
    0.17-0.34: fade in  (~2 sec start for 12 sec GIF)
    0.34-0.72: layer on
    0.72-0.90: fade out
    0.90-1.00: layer off
    """
    fade_in = smoothstep(0.17, 0.34, t)
    fade_out = 1.0 - smoothstep(0.72, 0.90, t)
    return fade_in * fade_out

# =========================
# Animation update
# =========================

def update(frame):
    t = frame / (FRAMES - 1)
    a = layer_alpha(t)

    pulse = 0.5 + 0.5 * np.sin(2 * np.pi * (t * 6))
    ring_scale = 1.0 + 0.26 * pulse

    for idx, (
        (glow, body, core, is_xxxvi, size),
        (ring1, ring2, base_size),
        label,
        connector
    ) in enumerate(zip(satellite_artists, ring_artists, label_artists, connector_artists)):

        # Random-looking staggered reveal
        delay = (idx % 7) * 0.018 + (idx // 7) * 0.006
        local_a = a * smoothstep(0.0, 0.13, max(t - delay, 0))

        boost = 1.35 if is_xxxvi else 1.0

        glow.set_alpha(0.18 * boost * local_a)
        body.set_alpha(0.72 * local_a)
        core.set_alpha(0.62 * local_a)

        ring1.set_radius(base_size * 2.0 * ring_scale)
        ring2.set_radius(base_size * 3.0 * (1.0 + 0.36 * pulse))

        ring1.set_alpha((0.48 + 0.26 * pulse) * boost * local_a)
        ring2.set_alpha((0.22 + 0.24 * (1 - pulse)) * boost * local_a)

        label_alpha = local_a * smoothstep(0.38, 0.54, t)
        connector_alpha = label_alpha * 0.72

        label.set_alpha(label_alpha)
        connector.set_alpha(connector_alpha)

    if a > 0.08:
        subtitle.set_text("faint dark-matter-dominated satellite galaxies highlighted")
        subtitle.set_color(CYAN)
        status_box.set_text("DWARF LAYER: ON")
        status_box.set_color(CYAN)
        counter.set_text(f"{N_SATELLITES} satellites highlighted")
        counter.set_color(CYAN)
    else:
        subtitle.set_text("satellite layer OFF")
        subtitle.set_color(MUTED)
        status_box.set_text("DWARF LAYER: OFF")
        status_box.set_color(MUTED)
        counter.set_text("0 satellites highlighted")
        counter.set_color(MUTED)

    return (
        [subtitle, status_box, counter]
        + [artist for group in satellite_artists for artist in group[:3]]
        + [artist for group in ring_artists for artist in group[:2]]
        + label_artists
        + connector_artists
    )

# =========================
# Save GIF
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

# Galaxy spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "andromeda_dark_dwarf_spectrum.gif"

FPS = 24
DURATION_SEC = 12
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#030711"
GRID = "#16324a"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
DEEP_BLUE = "#0b63ff"
ORANGE = "#ff9a3c"
MAGENTA = "#ff4dff"
WHITE = "#f0f6ff"
GREEN = "#48ffb3"

rng = np.random.default_rng(236)

# =========================
# Synthetic dark dwarf spectrum
# =========================

wl = np.linspace(3800, 7600, 1600)  # Angstrom

def gaussian(x, mu, amp, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

# Very faint old stellar continuum: red, low surface brightness
continuum = (
    0.075
    + 0.030 * (wl - wl.min()) / (wl.max() - wl.min())
    + 0.006 * np.sin((wl - 3800) / 420)
)

# Weak absorption features typical for old stellar populations
absorption_lines = [
    (3934, -0.022, 10, "Ca K"),
    (3969, -0.018, 12, "Ca H"),
    (4304, -0.014, 18, "G band"),
    (4861, -0.010, 16, "Hβ abs."),
    (5175, -0.018, 18, "Mg b"),
    (5892, -0.012, 16, "Na D"),
    (6563, -0.008, 18, "Hα weak"),
]

true_spectrum = continuum.copy()
for mu, amp, sigma, _ in absorption_lines:
    true_spectrum += gaussian(wl, mu, amp, sigma)

# Detector / sky noise dominates
noise_sigma = 0.030
single_exposure_noise = rng.normal(0, noise_sigma, size=wl.size)
observed = true_spectrum + single_exposure_noise

# Smoothed / stacked spectrum as if after integration
kernel_size = 31
kernel = np.ones(kernel_size) / kernel_size
stacked = np.convolve(observed, kernel, mode="same")

# Residual noise estimate
noise_floor = np.full_like(wl, noise_sigma)

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.14)

ax.set_xlim(wl.min(), wl.max())
ax.set_ylim(-0.045, 0.185)

ax.grid(True, color=GRID, linestyle=":", linewidth=1.0, alpha=0.75)

for spine in ax.spines.values():
    spine.set_color("#95a3b5")
    spine.set_linewidth(1.1)

ax.tick_params(colors=TEXT, labelsize=13, length=5)

ax.set_xlabel(r"Observed wavelength  $\lambda$  (Å)", color=TEXT, fontsize=18, labelpad=14)
ax.set_ylabel("Relative flux", color=TEXT, fontsize=18, labelpad=14)

fig.text(
    0.5,
    0.945,
    "DARK DWARF SATELLITE — FAINT STELLAR SPECTRUM",
    ha="center",
    va="center",
    color="#d7dde8",
    fontsize=23,
    fontweight="bold"
)

fig.text(
    0.5,
    0.905,
    "low surface brightness • weak absorption lines • signal hidden near the noise floor",
    ha="center",
    va="center",
    color=MUTED,
    fontsize=13,
    fontweight="bold"
)

# =========================
# Static guides
# =========================

ax.axhspan(
    -noise_sigma,
    noise_sigma,
    color=MAGENTA,
    alpha=0.08,
    zorder=0
)

ax.text(
    wl.min() + 70,
    noise_sigma + 0.004,
    "noise-dominated region",
    color=MAGENTA,
    fontsize=11,
    fontweight="bold",
    ha="left",
    va="bottom"
)

line_guides = []
line_texts = []

for mu, amp, sigma, label in absorption_lines:
    guide = ax.axvline(
        mu,
        color=CYAN,
        linestyle=":",
        linewidth=1.0,
        alpha=0.18,
        zorder=2
    )
    txt = ax.text(
        mu,
        0.172,
        label,
        color=CYAN,
        fontsize=9,
        fontweight="bold",
        ha="center",
        va="top",
        rotation=90,
        alpha=0.45,
        zorder=3
    )

    line_guides.append(guide)
    line_texts.append(txt)

info = (
    "schematic spectrum\n"
    "old metal-poor stars\n"
    "very low continuum\n"
    "no strong emission lines"
)

ax.text(
    0.97,
    0.92,
    info,
    transform=ax.transAxes,
    color=ORANGE,
    fontsize=11,
    ha="right",
    va="top",
    bbox=dict(
        boxstyle="round,pad=0.45",
        facecolor="#07111f",
        edgecolor=ORANGE,
        alpha=0.86
    ),
    zorder=30
)

# =========================
# Animated artists
# =========================

noise_glow, = ax.plot([], [], color=MAGENTA, linewidth=5, alpha=0.06, zorder=8)
noise_line, = ax.plot([], [], color=MAGENTA, linewidth=0.9, alpha=0.50, zorder=9)

true_line, = ax.plot([], [], color=MUTED, linewidth=1.2, alpha=0.0, zorder=10)

stacked_glow, = ax.plot([], [], color=CYAN, linewidth=8, alpha=0.0, zorder=12)
stacked_line, = ax.plot([], [], color=CYAN, linewidth=2.2, alpha=0.0, zorder=13)

continuum_line, = ax.plot([], [], color=GREEN, linewidth=1.4, alpha=0.0, zorder=14)

cursor = ax.scatter(
    [],
    [],
    s=70,
    color=WHITE,
    edgecolor=CYAN,
    linewidths=0.8,
    alpha=0.0,
    zorder=20
)

feature_markers = []
feature_labels = []

for mu, amp, sigma, label in absorption_lines:
    y = np.interp(mu, wl, true_spectrum)

    marker = ax.scatter(
        [],
        [],
        s=130,
        color=CYAN,
        edgecolor=WHITE,
        linewidths=0.8,
        alpha=0.0,
        zorder=22
    )

    label_artist = ax.text(
        mu + 35,
        y - 0.018,
        "weak absorption",
        color=CYAN,
        fontsize=9,
        fontweight="bold",
        alpha=0.0,
        zorder=23
    )

    feature_markers.append((marker, mu, y))
    feature_labels.append(label_artist)

status = ax.text(
    0.04,
    0.06,
    "spectrum: single noisy exposure",
    transform=ax.transAxes,
    color=MUTED,
    fontsize=12,
    ha="left",
    va="bottom",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358",
        alpha=0.85
    ),
    zorder=30
)

# =========================
# Animation helpers
# =========================

def smoothstep(edge0, edge1, value):
    t = np.clip((value - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

def ease(t):
    return 1 - (1 - t) ** 3

def layer_alpha(t):
    fade_in = smoothstep(0.04, 0.14, t)
    fade_out = 1.0 - smoothstep(0.92, 0.99, t)
    return fade_in * fade_out

# =========================
# Animation update
# =========================

def update(frame):
    t = frame / (FRAMES - 1)
    a = layer_alpha(t)

    scan = ease(t)
    xmax = wl.min() + scan * (wl.max() - wl.min())

    visible = wl <= xmax
    xv = wl[visible]

    # fresh noise slowly morphing frame-to-frame
    dynamic_noise = rng.normal(0, noise_sigma, size=wl.size)
    dynamic_observed = true_spectrum + dynamic_noise
    dynamic_stacked = np.convolve(dynamic_observed, kernel, mode="same")

    noise_line.set_data(xv, dynamic_observed[visible])
    noise_line.set_alpha(0.46 * a)

    noise_glow.set_data(xv, dynamic_observed[visible])
    noise_glow.set_alpha(0.055 * a)

    # hidden physical spectrum appears faintly as reference
    true_reveal = smoothstep(0.28, 0.48, t)
    true_line.set_data(xv, true_spectrum[visible])
    true_line.set_alpha(0.42 * a * true_reveal)

    # stacked/smoothed spectrum appears later
    stack_reveal = smoothstep(0.42, 0.65, t)
    stacked_line.set_data(xv, dynamic_stacked[visible])
    stacked_line.set_alpha(0.92 * a * stack_reveal)

    stacked_glow.set_data(xv, dynamic_stacked[visible])
    stacked_glow.set_alpha(0.13 * a * stack_reveal)

    continuum_line.set_data(xv, continuum[visible])
    continuum_line.set_alpha(0.55 * a * smoothstep(0.55, 0.72, t))

    # cursor
    if len(xv) > 0:
        y_cursor = np.interp(xv[-1], wl, dynamic_observed)
        cursor.set_offsets([[xv[-1], y_cursor]])
        cursor.set_alpha(0.85 * a)

    # feature markers reveal after scan passes each line
    pulse = 0.5 + 0.5 * np.sin(2 * np.pi * 6 * t)

    for idx, ((marker, mu, y), label_artist) in enumerate(zip(feature_markers, feature_labels)):
        passed = smoothstep(mu - 40, mu + 80, xmax)
        local = a * stack_reveal * passed

        marker.set_offsets([[mu, y]])
        marker.set_alpha((0.32 + 0.35 * pulse) * local)
        marker.set_sizes([90 + 80 * pulse])

        # not every label at once; avoids clutter
        if idx in {0, 2, 4, 6}:
            label_artist.set_alpha(0.75 * local)
        else:
            label_artist.set_alpha(0.0)

    if stack_reveal < 0.25:
        status.set_text("spectrum: single noisy exposure")
        status.set_color(MUTED)
    elif stack_reveal < 0.85:
        status.set_text("integration: weak stellar continuum emerging")
        status.set_color(CYAN)
    else:
        status.set_text("diagnostic: faint absorption features identified")
        status.set_color(GREEN)

    return (
        noise_line,
        noise_glow,
        true_line,
        stacked_line,
        stacked_glow,
        continuum_line,
        cursor,
        status,
        *[m[0] for m in feature_markers],
        *feature_labels
    )

# =========================
# Save GIF
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Ellipse, Circle
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "andromeda_xxxvi_profile_reveal.gif"

FPS = 24
DURATION_SEC = 16
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#020510"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
DEEP_BLUE = "#0b63ff"
ORANGE = "#ff9a3c"
WHITE = "#f0f6ff"
GREEN = "#48ffb3"

rng = np.random.default_rng(360)

# =========================
# Scene geometry
# =========================

W, H = 16, 9
N_STARS = 420
N_FAINT_GALAXIES = 65

# M31 and dwarf positions
m31_x, m31_y = 7.2, 4.75
dwarf_x, dwarf_y = 10.95, 3.35

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_position([0, 0, 1, 1])

PAD = 0.18
ax.set_xlim(PAD, W - PAD)
ax.set_ylim(PAD, H - PAD)
ax.set_aspect("equal")
ax.axis("off")

# =========================
# Background stars
# =========================

star_x = rng.uniform(0, W, N_STARS)
star_y = rng.uniform(0, H, N_STARS)
star_s = rng.uniform(0.5, 7.0, N_STARS)
star_alpha = rng.uniform(0.10, 0.68, N_STARS)

ax.scatter(
    star_x,
    star_y,
    s=star_s,
    c=WHITE,
    alpha=star_alpha,
    linewidths=0,
    zorder=1
)

# brighter stars
for _ in range(14):
    sx = rng.uniform(0.5, W - 0.5)
    sy = rng.uniform(0.5, H - 0.5)
    ss = rng.uniform(12, 26)
    ax.scatter([sx], [sy], s=ss, c=WHITE, alpha=0.70, linewidths=0, zorder=2)
    ax.plot([sx - 0.07, sx + 0.07], [sy, sy], color=WHITE, alpha=0.22, linewidth=0.7, zorder=2)
    ax.plot([sx, sx], [sy - 0.07, sy + 0.07], color=WHITE, alpha=0.22, linewidth=0.7, zorder=2)

# faint background galaxies
palette = ["#f7d9a0", "#d7c7ff", "#ffb27a", "#dce7ff", "#b7c7ff", "#ffd0c2"]

for _ in range(N_FAINT_GALAXIES):
    gx = rng.uniform(0.5, W - 0.5)
    gy = rng.uniform(0.5, H - 0.5)

    if (gx - m31_x) ** 2 + (gy - m31_y) ** 2 < 4.5:
        continue

    width = rng.uniform(0.035, 0.16)
    height = width * rng.uniform(0.25, 0.8)
    angle = rng.uniform(0, 180)
    color = rng.choice(palette)
    alpha = rng.uniform(0.10, 0.32)

    ax.add_patch(Ellipse(
        (gx, gy),
        width * 2.0,
        height * 2.0,
        angle=angle,
        facecolor=color,
        edgecolor="none",
        alpha=alpha * 0.12,
        zorder=2
    ))

    ax.add_patch(Ellipse(
        (gx, gy),
        width,
        height,
        angle=angle,
        facecolor=color,
        edgecolor="none",
        alpha=alpha,
        zorder=3
    ))

# =========================
# M31 schematic
# =========================

m31_angle = -13

for scale, alpha, color in [
    (4.7, 0.032, "#7da8ff"),
    (3.6, 0.052, "#89b6ff"),
    (2.65, 0.078, "#b7c7ff"),
]:
    ax.add_patch(Ellipse(
        (m31_x, m31_y),
        width=scale,
        height=scale * 0.33,
        angle=m31_angle,
        facecolor=color,
        edgecolor="none",
        alpha=alpha,
        zorder=5
    ))

for scale, alpha, color in [
    (3.0, 0.22, "#6f91d8"),
    (2.3, 0.26, "#9eb5e8"),
    (1.55, 0.34, "#d7d3c7"),
    (0.78, 0.68, "#ffd9a4"),
]:
    ax.add_patch(Ellipse(
        (m31_x, m31_y),
        width=scale,
        height=scale * 0.30,
        angle=m31_angle,
        facecolor=color,
        edgecolor="none",
        alpha=alpha,
        zorder=7
    ))

ax.add_patch(Circle((m31_x, m31_y), 0.34, facecolor="#ffe7b7", edgecolor="none", alpha=0.24, zorder=8))
ax.add_patch(Circle((m31_x, m31_y), 0.13, facecolor="#fff1d0", edgecolor="none", alpha=0.88, zorder=9))

for offset, scale, alpha in [
    (-0.10, 2.35, 0.18),
    (0.08, 1.95, 0.14),
    (0.22, 1.50, 0.12),
]:
    ax.add_patch(Ellipse(
        (m31_x + offset, m31_y + offset * 0.1),
        width=scale,
        height=scale * 0.22,
        angle=m31_angle,
        facecolor="none",
        edgecolor="#1b2438",
        linewidth=1.3,
        alpha=alpha,
        zorder=10
    ))

m31_label = ax.text(
    m31_x - 1.0,
    m31_y - 0.78,
    "M31 / Andromeda Galaxy",
    color=TEXT,
    fontsize=11,
    fontweight="bold",
    alpha=0.72,
    zorder=30
)

# =========================
# Dwarf galaxy: faint diffuse satellite
# =========================

dwarf_glow = Ellipse(
    (dwarf_x, dwarf_y),
    0.72,
    0.42,
    angle=24,
    facecolor=CYAN,
    edgecolor="none",
    alpha=0.08,
    zorder=14
)

dwarf_body = Ellipse(
    (dwarf_x, dwarf_y),
    0.30,
    0.18,
    angle=24,
    facecolor="#8edcff",
    edgecolor="none",
    alpha=0.22,
    zorder=15
)

dwarf_core = Circle(
    (dwarf_x, dwarf_y),
    0.035,
    facecolor=WHITE,
    edgecolor="none",
    alpha=0.38,
    zorder=16
)

ax.add_patch(dwarf_glow)
ax.add_patch(dwarf_body)
ax.add_patch(dwarf_core)

# Distance line to M31
distance_line, = ax.plot(
    [m31_x, dwarf_x],
    [m31_y, dwarf_y],
    color=CYAN,
    linewidth=1.0,
    linestyle=(0, (4, 5)),
    alpha=0.0,
    zorder=18
)

distance_label = ax.text(
    (m31_x + dwarf_x) / 2 - 0.2,
    (m31_y + dwarf_y) / 2 - 1.75,
    "388,000 light years from M31",
    color=CYAN,
    fontsize=10,
    fontweight="bold",
    alpha=0.0,
    zorder=30
)

# =========================
# Target reticle
# =========================

reticle_r1 = Circle(
    (dwarf_x, dwarf_y),
    0.38,
    facecolor="none",
    edgecolor=CYAN,
    linewidth=1.5,
    alpha=0.0,
    zorder=24
)

reticle_r2 = Circle(
    (dwarf_x, dwarf_y),
    0.56,
    facecolor="none",
    edgecolor=DEEP_BLUE,
    linewidth=0.9,
    alpha=0.0,
    zorder=24
)

ax.add_patch(reticle_r1)
ax.add_patch(reticle_r2)

reticle_h1, = ax.plot([], [], color=CYAN, linewidth=1.0, alpha=0.0, zorder=25)
reticle_h2, = ax.plot([], [], color=CYAN, linewidth=1.0, alpha=0.0, zorder=25)
reticle_v1, = ax.plot([], [], color=CYAN, linewidth=1.0, alpha=0.0, zorder=25)
reticle_v2, = ax.plot([], [], color=CYAN, linewidth=1.0, alpha=0.0, zorder=25)

connector, = ax.plot(
    [dwarf_x + 0.45, 11.65],
    [dwarf_y + 0.15, 6.62],
    color=CYAN,
    linewidth=0.8,
    alpha=0.0,
    zorder=28
)

# =========================
# Text panel
# =========================

panel_x, panel_y = 11.72, 6.75

panel_title = ax.text(
    panel_x,
    panel_y,
    "Andromeda XXXVI",
    color=ORANGE,
    fontsize=18,
    fontweight="bold",
    ha="left",
    va="top",
    alpha=0.0,
    zorder=40
)

panel_subtitle = ax.text(
    panel_x,
    panel_y - 0.36,
    "ultra-faint dwarf satellite candidate",
    color=MUTED,
    fontsize=11,
    fontweight="bold",
    ha="left",
    va="top",
    alpha=0.0,
    zorder=40
)

facts = [
    "Distance to us: 2.53 mln lyrs",
    "Distance to Andromeda Galaxy: 388,000 lyrs",
    "Absolute magnitude: −6",
    "Radius: 200 ly",
    "Age: 12.5 bln yrs",
    "Metallicity: low",
]

fact_artists = []

for i, fact in enumerate(facts):
    txt = ax.text(
        panel_x,
        panel_y - 0.82 - i * 0.36,
        fact,
        color=TEXT if i < 5 else GREEN,
        fontsize=11.5,
        ha="left",
        va="top",
        alpha=0.0,
        zorder=40
    )
    fact_artists.append(txt)

status = ax.text(
    0.5,
    8.55,
    "M31 satellite field — target acquisition",
    color=TEXT,
    fontsize=18,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=40
)

phase_text = ax.text(
    0.5,
    8.22,
    "searching field",
    color=MUTED,
    fontsize=12,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=40
)

# subtle HUD frame
frame_style = dict(color="#2b4358", linewidth=1.0, alpha=0.42, zorder=35)
ax.plot([0.35, 3.0], [8.75, 8.75], **frame_style)
ax.plot([0.35, 0.35], [8.75, 7.95], **frame_style)
ax.plot([13.0, 15.65], [8.75, 8.75], **frame_style)
ax.plot([15.65, 15.65], [8.75, 7.95], **frame_style)
ax.plot([0.35, 2.1], [0.25, 0.25], **frame_style)
ax.plot([0.35, 0.35], [0.25, 0.95], **frame_style)
ax.plot([13.9, 15.65], [0.25, 0.25], **frame_style)
ax.plot([15.65, 15.65], [0.25, 0.95], **frame_style)

# =========================
# Animation helpers
# =========================

def smoothstep(edge0, edge1, x):
    if edge0 == edge1:
        return 1.0 if x >= edge1 else 0.0
    t = np.clip((x - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

def reveal_window(t, start, end):
    return smoothstep(start, start + 0.08, t) * (1.0 - smoothstep(end - 0.08, end, t))

def main_alpha(t):
    fade_in = smoothstep(0.03, 0.10, t)
    fade_out = 1.0 - smoothstep(0.97, 1.00, t)
    return fade_in * fade_out

# =========================
# Animation update
# =========================

def update(frame):
    t = frame / (FRAMES - 1)
    a = main_alpha(t)

    # Reticle appears after ~2 sec
    target_a = a * smoothstep(0.16, 0.32, t)
    distance_a = a * smoothstep(0.26, 0.42, t)

    pulse = 0.5 + 0.5 * np.sin(2 * np.pi * 6 * t)

    reticle_r1.set_alpha((0.42 + 0.28 * pulse) * target_a)
    reticle_r2.set_alpha((0.20 + 0.22 * (1 - pulse)) * target_a)

    reticle_r1.set_radius(0.34 + 0.045 * pulse)
    reticle_r2.set_radius(0.52 + 0.075 * pulse)

    arm_inner = 0.42
    arm_outer = 0.70 + 0.06 * pulse

    reticle_h1.set_data([dwarf_x - arm_outer, dwarf_x - arm_inner], [dwarf_y, dwarf_y])
    reticle_h2.set_data([dwarf_x + arm_inner, dwarf_x + arm_outer], [dwarf_y, dwarf_y])
    reticle_v1.set_data([dwarf_x, dwarf_x], [dwarf_y - arm_outer, dwarf_y - arm_inner])
    reticle_v2.set_data([dwarf_x, dwarf_x], [dwarf_y + arm_inner, dwarf_y + arm_outer])

    for arm in (reticle_h1, reticle_h2, reticle_v1, reticle_v2):
        arm.set_alpha(0.75 * target_a)

    dwarf_glow.set_alpha(0.08 + 0.12 * target_a * pulse)
    dwarf_body.set_alpha(0.22 + 0.22 * target_a)
    dwarf_core.set_alpha(0.38 + 0.25 * target_a)

    distance_line.set_alpha(0.45 * distance_a)
    distance_label.set_alpha(0.90 * distance_a)

    panel_a = a * smoothstep(0.34, 0.50, t)
    panel_title.set_alpha(panel_a)
    panel_subtitle.set_alpha(panel_a * 0.9)
    connector, = ax.plot([], [], alpha=0.0)

    for i, txt in enumerate(fact_artists):
        line_a = a * smoothstep(0.36 + i * 0.035, 0.43 + i * 0.035, t)
        txt.set_alpha(line_a)
    
    if t < 0.16:
        phase_text.set_text("searching field")
        phase_text.set_color(MUTED)
    elif t < 0.34:
        phase_text.set_text("target acquired: faint dwarf satellite")
        phase_text.set_color(CYAN)
    elif t < 0.72:
        phase_text.set_text("physical parameters revealed")
        phase_text.set_color(ORANGE)
    else:
        phase_text.set_text("ancient low-metallicity satellite system")
        phase_text.set_color(GREEN)

    return (
        reticle_r1,
        reticle_r2,
        reticle_h1,
        reticle_h2,
        reticle_v1,
        reticle_v2,
        dwarf_glow,
        dwarf_body,
        dwarf_core,
        distance_line,
        distance_label,
       # connector,
        panel_title,
        panel_subtitle,
        phase_text,
        *fact_artists,
    )

# =========================
# Save GIF
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Ellipse, Circle
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "andromeda_satellites_overlay_field.gif"

FPS = 24
DURATION_SEC = 14
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#020510"       # later can be keyed/removed if needed
CYAN = "#35c9ff"
DEEP_BLUE = "#0b63ff"
ORANGE = "#ff9a3c"
WHITE = "#f0f6ff"
MUTED = "#6f8194"

rng = np.random.default_rng(777)

# =========================
# Scene geometry
# =========================

W, H = 16, 9
N_SATELLITES = 28
UNKNOWN_FRACTION = 0.38

# =========================
# Figure setup
# =========================

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_position([0, 0, 1, 1])

PAD = 0.20
ax.set_xlim(PAD, W - PAD)
ax.set_ylim(PAD, H - PAD)
ax.set_aspect("equal")
ax.axis("off")

# =========================
# Helpers
# =========================

def roman(n):
    numerals = [
        (1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
        (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
        (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")
    ]
    out = ""
    for value, symbol in numerals:
        while n >= value:
            out += symbol
            n -= value
    return out

def smoothstep(edge0, edge1, x):
    if edge0 == edge1:
        return 1.0 if x >= edge1 else 0.0
    t = np.clip((x - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

# =========================
# Generate chaotic satellite positions
# =========================

positions = []

attempts = 0
while len(positions) < N_SATELLITES and attempts < 5000:
    attempts += 1

    x = rng.uniform(0.75, W - 0.75)
    y = rng.uniform(0.75, H - 0.75)

    # avoid too much overlap
    if all((x - px) ** 2 + (y - py) ** 2 > 0.55 for px, py in positions):
        positions.append((x, y))

# Random labels: some roman, some unknown
roman_pool = list(range(1, 38))
rng.shuffle(roman_pool)

labels = []
for i in range(N_SATELLITES):
    if rng.random() < UNKNOWN_FRACTION:
        labels.append("?")
    else:
        labels.append(roman(roman_pool.pop()))

# ensure at least one high-number known satellite
labels[0] = "XXXVI"

# random reveal order
reveal_order = np.arange(N_SATELLITES)
rng.shuffle(reveal_order)

reveal_rank = np.zeros(N_SATELLITES, dtype=int)
for rank, idx in enumerate(reveal_order):
    reveal_rank[idx] = rank

# =========================
# Artists
# =========================

satellite_artists = []
ring_artists = []
label_artists = []
connector_artists = []

for i, ((x, y), label_text) in enumerate(zip(positions, labels)):
    is_unknown = label_text == "?"
    is_special = label_text == "XXXVI"

    size = rng.uniform(0.075, 0.155)
    ratio = rng.uniform(0.65, 1.25)
    angle = rng.uniform(0, 180)

    color = ORANGE if is_special else (MUTED if is_unknown else CYAN)
    glow_color = ORANGE if is_special else (DEEP_BLUE if not is_unknown else MUTED)

    glow = Ellipse(
        (x, y),
        size * 6.5,
        size * 6.5 * ratio,
        angle=angle,
        facecolor=glow_color,
        edgecolor="none",
        alpha=0.0,
        zorder=10
    )

    body = Ellipse(
        (x, y),
        size,
        size * ratio,
        angle=angle,
        facecolor=color,
        edgecolor=WHITE if not is_unknown else MUTED,
        linewidth=0.35,
        alpha=0.0,
        zorder=11
    )

    core = Circle(
        (x, y),
        radius=size * 0.16,
        facecolor=WHITE,
        edgecolor="none",
        alpha=0.0,
        zorder=12
    )

    ring1 = Circle(
        (x, y),
        radius=size * 2.1,
        edgecolor=color,
        facecolor="none",
        linewidth=1.1,
        alpha=0.0,
        zorder=13
    )

    ring2 = Circle(
        (x, y),
        radius=size * 3.0,
        edgecolor=glow_color,
        facecolor="none",
        linewidth=0.8,
        alpha=0.0,
        zorder=13
    )

    # Label offset
    dx = rng.choice([-1, 1]) * rng.uniform(0.25, 0.50)
    dy = rng.choice([-1, 1]) * rng.uniform(0.18, 0.38)
    ha = "left" if dx > 0 else "right"

    label = ax.text(
        x + dx,
        y + dy,
        label_text,
        color=color,
        fontsize=10.5 if is_unknown else 9.2,
        fontweight="bold",
        ha=ha,
        va="center",
        alpha=0.0,
        zorder=15
    )

    connector, = ax.plot(
        [x, x + dx * 0.68],
        [y, y + dy * 0.68],
        color=color,
        linewidth=0.65,
        alpha=0.0,
        zorder=14
    )

    for artist in [glow, body, core, ring1, ring2]:
        ax.add_patch(artist)

    satellite_artists.append((glow, body, core, size, is_unknown, is_special))
    ring_artists.append((ring1, ring2, size))
    label_artists.append(label)
    connector_artists.append(connector)

# =========================
# Animation logic
# =========================

def object_alpha(t, rank):
    """
    Each object appears in random order and then stays visible.
    """
    start = 0.08 + rank * (0.58 / max(N_SATELLITES - 1, 1))
    end = start + 0.10
    return smoothstep(start, end, t)

def global_alpha(t):
    """
    Fade in at start, hold almost to the end, then soft fade out.
    """
    fade_in = smoothstep(0.02, 0.08, t)
    fade_out = 1.0 - smoothstep(0.92, 0.99, t)
    return fade_in * fade_out

def update(frame):
    t = frame / (FRAMES - 1)
    g = global_alpha(t)

    pulse = 0.5 + 0.5 * np.sin(2 * np.pi * 6 * t)

    for i, (
        (glow, body, core, size, is_unknown, is_special),
        (ring1, ring2, base_size),
        label,
        connector
    ) in enumerate(zip(satellite_artists, ring_artists, label_artists, connector_artists)):

        a = g * object_alpha(t, reveal_rank[i])

        boost = 1.35 if is_special else 1.0
        unknown_dim = 0.70 if is_unknown else 1.0

        glow.set_alpha(0.18 * boost * unknown_dim * a)
        body.set_alpha(0.78 * unknown_dim * a)
        core.set_alpha(0.60 * unknown_dim * a)

        ring1.set_radius(base_size * (2.0 + 0.28 * pulse))
        ring2.set_radius(base_size * (3.0 + 0.45 * (1 - pulse)))

        ring1.set_alpha((0.42 + 0.26 * pulse) * boost * unknown_dim * a)
        ring2.set_alpha((0.20 + 0.22 * (1 - pulse)) * boost * unknown_dim * a)

        # label appears slightly after object body
        label_a = g * smoothstep(0.02, 0.12, object_alpha(t, reveal_rank[i]))
        label.set_alpha(0.95 * label_a)
        # connector.set_alpha(0.70 * label_a)

    return (
        [artist for group in satellite_artists for artist in group[:3]]
        + [artist for group in ring_artists for artist in group[:2]]
        + label_artists
        + connector_artists
    )

# =========================
# Save GIF
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

# Machine Learning Illustration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "llm_training_concept_dark.gif"

FPS = 24
DURATION_SEC = 14
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#020510"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
BLUE = "#0b63ff"
ORANGE = "#ff9a3c"
MAGENTA = "#ff4dff"
GREEN = "#48ffb3"
WHITE = "#f0f6ff"
RED = "#ff5a45"

rng = np.random.default_rng(512)

# =========================
# Scene geometry
# =========================

W, H = 16, 9

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_position([0, 0, 1, 1])

ax.set_xlim(0, W)
ax.set_ylim(0, H)
ax.set_aspect("equal")
ax.axis("off")

# =========================
# Helpers
# =========================

def smoothstep(edge0, edge1, x):
    if edge0 == edge1:
        return 1.0 if x >= edge1 else 0.0
    t = np.clip((x - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

def pulse(t, freq=6, phase=0):
    return 0.5 + 0.5 * np.sin(2 * np.pi * (freq * t + phase))

def arrow(x1, y1, x2, y2, color, alpha=0.65, zorder=10):
    a = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="->",
        mutation_scale=14,
        linewidth=1.4,
        color=color,
        alpha=alpha,
        zorder=zorder
    )
    ax.add_patch(a)
    return a

def make_box(x, y, w, h, label, color, fontsize=11, zorder=10):
    rect = Rectangle(
        (x, y),
        w,
        h,
        facecolor="#07111f",
        edgecolor=color,
        linewidth=1.1,
        alpha=0.88,
        zorder=zorder
    )
    ax.add_patch(rect)

    txt = ax.text(
        x + w / 2,
        y + h / 2,
        label,
        color=color,
        fontsize=fontsize,
        fontweight="bold",
        ha="center",
        va="center",
        zorder=zorder + 1
    )
    return rect, txt

# =========================
# Background particles
# =========================

star_x = rng.uniform(0, W, 180)
star_y = rng.uniform(0, H, 180)
star_s = rng.uniform(0.5, 5.0, 180)
star_a = rng.uniform(0.08, 0.35, 180)

ax.scatter(
    star_x,
    star_y,
    s=star_s,
    c=WHITE,
    alpha=star_a,
    linewidths=0,
    zorder=1
)

# =========================
# Title
# =========================

title = ax.text(
    0.55,
    8.55,
    "LLM TRAINING — NEXT TOKEN PREDICTION",
    color=TEXT,
    fontsize=19,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=40
)

subtitle = ax.text(
    0.55,
    8.22,
    "tokens → embeddings → transformer layers → loss → gradient update",
    color=MUTED,
    fontsize=12,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=40
)

# =========================
# Pipeline boxes
# =========================

token_box, token_txt = make_box(0.75, 6.25, 2.35, 0.78, "TRAINING TEXT", CYAN)
embed_box, embed_txt = make_box(3.65, 6.25, 2.15, 0.78, "EMBEDDINGS", BLUE)
transformer_box, transformer_txt = make_box(6.35, 5.30, 3.65, 2.65, "TRANSFORMER\nBLOCKS", ORANGE, fontsize=14)
logits_box, logits_txt = make_box(10.65, 6.25, 2.05, 0.78, "LOGITS", MAGENTA)
loss_box, loss_txt = make_box(13.15, 6.25, 2.05, 0.78, "LOSS", RED)

arrows = [
    arrow(3.10, 6.64, 3.62, 6.64, CYAN),
    arrow(5.80, 6.64, 6.32, 6.64, BLUE),
    arrow(10.02, 6.64, 10.62, 6.64, ORANGE),
    arrow(12.70, 6.64, 13.12, 6.64, MAGENTA),
]

# gradient update loop arrow
grad_arrow = FancyArrowPatch(
    (14.05, 6.20),
    (8.15, 4.95),
    connectionstyle="arc3,rad=-0.25",
    arrowstyle="->",
    mutation_scale=18,
    linewidth=1.5,
    color=GREEN,
    alpha=0.0,
    zorder=12
)
ax.add_patch(grad_arrow)

grad_label = ax.text(
    10.9,
    4.70,
    "gradient update",
    color=GREEN,
    fontsize=11,
    fontweight="bold",
    alpha=0.0,
    ha="center",
    va="center",
    zorder=20
)

# =========================
# Token stream
# =========================

tokens = ["The", "galaxy", "is", "very", "faint", "because", "it", "contains", "old", "stars"]
token_artists = []

for i, tok in enumerate(tokens):
    x = 0.88 + (i % 5) * 0.40
    y = 5.78 - (i // 5) * 0.32

    rect = Rectangle(
        (x, y),
        0.34,
        0.22,
        facecolor="#07111f",
        edgecolor=CYAN,
        linewidth=0.7,
        alpha=0.0,
        zorder=15
    )
    ax.add_patch(rect)

    txt = ax.text(
        x + 0.17,
        y + 0.11,
        tok[:3],
        color=CYAN,
        fontsize=6.5,
        fontweight="bold",
        ha="center",
        va="center",
        alpha=0.0,
        zorder=16
    )

    token_artists.append((rect, txt))

# =========================
# Embedding vectors
# =========================

embedding_lines = []

for i in range(9):
    y = 5.65 + i * 0.12
    line, = ax.plot(
        [3.85, 5.45],
        [y, y],
        color=BLUE,
        linewidth=1.1,
        alpha=0.0,
        zorder=15
    )
    embedding_lines.append(line)

embedding_dots = []
for i in range(9):
    dot = ax.scatter(
        [],
        [],
        s=26,
        color=CYAN,
        alpha=0.0,
        zorder=18
    )
    embedding_dots.append(dot)

# =========================
# Transformer block internals
# =========================

layer_rects = []
layer_labels = []
layer_nodes = []

layer_names = ["ATTENTION", "MLP", "NORM", "ATTENTION", "MLP"]

for i, name in enumerate(layer_names):
    y = 7.32 - i * 0.42

    rect = Rectangle(
        (6.72, y),
        2.92,
        0.28,
        facecolor="#101b2d",
        edgecolor=ORANGE,
        linewidth=0.8,
        alpha=0.75,
        zorder=16
    )
    ax.add_patch(rect)

    txt = ax.text(
        8.18,
        y + 0.14,
        name,
        color=ORANGE,
        fontsize=8.5,
        fontweight="bold",
        ha="center",
        va="center",
        alpha=0.78,
        zorder=17
    )

    layer_rects.append(rect)
    layer_labels.append(txt)

# attention matrix
attn_grid = []
grid_x0, grid_y0 = 7.02, 5.55
cell = 0.16

for iy in range(8):
    row = []
    for ix in range(8):
        r = Rectangle(
            (grid_x0 + ix * cell, grid_y0 + iy * cell),
            cell * 0.85,
            cell * 0.85,
            facecolor=CYAN,
            edgecolor="none",
            alpha=0.04,
            zorder=19
        )
        ax.add_patch(r)
        row.append(r)
    attn_grid.append(row)

attn_label = ax.text(
    grid_x0 + 0.62,
    grid_y0 - 0.22,
    "attention weights",
    color=MUTED,
    fontsize=8,
    ha="center",
    va="center",
    alpha=0.65,
    zorder=20
)

# =========================
# Prediction panel
# =========================

pred_title = ax.text(
    10.80,
    5.55,
    "next token probabilities",
    color=TEXT,
    fontsize=11,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=20
)

pred_tokens = ["stars", "dust", "gas", "noise"]
pred_bars = []
pred_labels = []

for i, tok in enumerate(pred_tokens):
    y = 5.18 - i * 0.32

    label = ax.text(
        10.82,
        y,
        tok,
        color=TEXT,
        fontsize=9,
        ha="left",
        va="center",
        zorder=20
    )

    bar_bg = Rectangle(
        (11.55, y - 0.06),
        1.15,
        0.12,
        facecolor="#101b2d",
        edgecolor="#2b4358",
        linewidth=0.5,
        alpha=0.9,
        zorder=19
    )
    ax.add_patch(bar_bg)

    bar = Rectangle(
        (11.55, y - 0.06),
        0.10,
        0.12,
        facecolor=CYAN if tok == "stars" else MAGENTA,
        edgecolor="none",
        alpha=0.85,
        zorder=20
    )
    ax.add_patch(bar)

    pred_labels.append(label)
    pred_bars.append(bar)

# =========================
# Loss panel
# =========================

loss_curve_x = np.linspace(13.30, 15.05, 80)
loss_curve_y = 5.35 + 0.95 * np.exp(-np.linspace(0, 3.3, 80))

loss_glow, = ax.plot([], [], color=RED, linewidth=7, alpha=0.0, zorder=18)
loss_line, = ax.plot([], [], color=RED, linewidth=2.0, alpha=0.0, zorder=19)

loss_value = ax.text(
    14.18,
    5.18,
    "loss: --",
    color=RED,
    fontsize=11,
    fontweight="bold",
    ha="center",
    va="center",
    zorder=22
)

# =========================
# Weights panel
# =========================

weights_title = ax.text(
    0.85,
    3.95,
    "MODEL WEIGHTS",
    color=TEXT,
    fontsize=12,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=20
)

weight_nodes = []
weight_edges = []

node_positions = []
for col, x0 in enumerate([1.0, 1.8, 2.6, 3.4]):
    n_nodes = 5
    for j in range(n_nodes):
        y0 = 2.35 + j * 0.28 + (col % 2) * 0.06
        node_positions.append((col, x0, y0))

for col, x0, y0 in node_positions:
    c = Circle(
        (x0, y0),
        0.045,
        facecolor=CYAN,
        edgecolor=WHITE,
        linewidth=0.4,
        alpha=0.45,
        zorder=17
    )
    ax.add_patch(c)
    weight_nodes.append(c)

for col in range(3):
    left = [(x, y) for c, x, y in node_positions if c == col]
    right = [(x, y) for c, x, y in node_positions if c == col + 1]

    for lx, ly in left:
        for rx, ry in right:
            if rng.random() < 0.45:
                line, = ax.plot(
                    [lx, rx],
                    [ly, ry],
                    color=CYAN,
                    linewidth=rng.uniform(0.35, 0.9),
                    alpha=0.10,
                    zorder=16
                )
                weight_edges.append(line)

# =========================
# Status text
# =========================

stage_text = ax.text(
    0.75,
    0.70,
    "stage: read training sequence",
    color=MUTED,
    fontsize=12,
    ha="left",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="#07111f",
        edgecolor="#2b4358",
        alpha=0.85
    ),
    zorder=50
)

# =========================
# Animation update
# =========================

def update(frame):
    t = frame / (FRAMES - 1)

    global_alpha = smoothstep(0.02, 0.08, t) * (1.0 - smoothstep(0.94, 0.99, t))
    p = pulse(t, 6)

    # token reveal
    token_phase = smoothstep(0.04, 0.24, t)
    for i, (rect, txt) in enumerate(token_artists):
        local = smoothstep(i / len(tokens), i / len(tokens) + 0.18, token_phase)
        rect.set_alpha(0.82 * local * global_alpha)
        txt.set_alpha(0.95 * local * global_alpha)

    # embeddings
    embed_a = smoothstep(0.18, 0.34, t) * global_alpha
    for i, line in enumerate(embedding_lines):
        line.set_alpha((0.32 + 0.30 * pulse(t, 3, i * 0.08)) * embed_a)
        line.set_linewidth(0.9 + 1.2 * pulse(t, 4, i * 0.05))

    for i, dot in enumerate(embedding_dots):
        xdot = 3.85 + 1.60 * ((t * 1.6 + i * 0.09) % 1.0)
        ydot = 5.65 + i * 0.12
        dot.set_offsets([[xdot, ydot]])
        dot.set_alpha(0.85 * embed_a)

    # transformer processing
    trans_a = smoothstep(0.28, 0.52, t) * global_alpha
    for i, rect in enumerate(layer_rects):
        intensity = 0.45 + 0.55 * pulse(t, 4, i * 0.17)
        rect.set_alpha((0.55 + 0.25 * intensity) * trans_a)

    for iy, row in enumerate(attn_grid):
        for ix, cell_rect in enumerate(row):
            diagonal = np.exp(-0.5 * ((ix - iy) / 1.6) ** 2)
            flicker = pulse(t, 5, (ix * 0.07 + iy * 0.05))
            cell_rect.set_alpha((0.04 + 0.42 * diagonal * flicker) * trans_a)

    # prediction bars improve over time
    pred_a = smoothstep(0.44, 0.64, t) * global_alpha
    confidence = smoothstep(0.48, 0.86, t)

    probs_start = np.array([0.27, 0.25, 0.24, 0.24])
    probs_end = np.array([0.74, 0.11, 0.09, 0.06])
    probs = probs_start * (1 - confidence) + probs_end * confidence

    for i, bar in enumerate(pred_bars):
        bar.set_width(1.15 * probs[i])
        bar.set_alpha((0.55 + 0.30 * p) * pred_a if i == 0 else 0.55 * pred_a)

    for lab in pred_labels:
        lab.set_alpha(pred_a)

    pred_title.set_alpha(pred_a)

    # loss curve
    loss_a = smoothstep(0.52, 0.76, t) * global_alpha
    n = int(1 + (len(loss_curve_x) - 1) * smoothstep(0.52, 0.88, t))

    loss_line.set_data(loss_curve_x[:n], loss_curve_y[:n])
    loss_glow.set_data(loss_curve_x[:n], loss_curve_y[:n])

    loss_line.set_alpha(0.95 * loss_a)
    loss_glow.set_alpha(0.13 * loss_a)

    current_loss = 2.8 - 1.9 * confidence + 0.08 * np.sin(2 * np.pi * 5 * t)
    loss_value.set_text(f"loss: {current_loss:.2f}")
    loss_value.set_alpha(loss_a)

    # gradient update
    grad_a = smoothstep(0.68, 0.82, t) * (1.0 - smoothstep(0.90, 0.98, t)) * global_alpha
    grad_arrow.set_alpha(0.75 * grad_a)
    grad_label.set_alpha(0.9 * grad_a)

    # weights pulse after gradient update
    update_a = smoothstep(0.70, 0.94, t) * global_alpha

    for i, node in enumerate(weight_nodes):
        node.set_alpha((0.35 + 0.45 * pulse(t, 6, i * 0.03)) * max(global_alpha, update_a))
        node.set_radius(0.042 + 0.018 * update_a * pulse(t, 5, i * 0.09))

    for i, edge in enumerate(weight_edges):
        edge.set_alpha((0.08 + 0.22 * update_a * pulse(t, 4, i * 0.04)) * global_alpha)

    # status
    if t < 0.22:
        stage_text.set_text("stage: read training sequence")
        stage_text.set_color(CYAN)
    elif t < 0.42:
        stage_text.set_text("stage: convert tokens into vectors")
        stage_text.set_color(BLUE)
    elif t < 0.62:
        stage_text.set_text("stage: transformer computes context")
        stage_text.set_color(ORANGE)
    elif t < 0.78:
        stage_text.set_text("stage: compare prediction with target token")
        stage_text.set_color(RED)
    else:
        stage_text.set_text("stage: update weights and reduce loss")
        stage_text.set_color(GREEN)

    return (
        [stage_text, grad_arrow, grad_label, loss_line, loss_glow, loss_value, pred_title]
        + [a for pair in token_artists for a in pair]
        + embedding_lines
        + embedding_dots
        + layer_rects
        + [cell for row in attn_grid for cell in row]
        + pred_bars
        + pred_labels
        + weight_nodes
        + weight_edges
    )

# =========================
# Save GIF
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")

# Neuro Networks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Circle, FancyArrowPatch
from IPython.display import Image, display
from pathlib import Path

# =========================
# Config
# =========================

OUT_DIR = Path("animations")
OUT_DIR.mkdir(exist_ok=True)

GIF_PATH = OUT_DIR / "neural_network_layers_training.gif"

FPS = 24
DURATION_SEC = 12
FRAMES = FPS * DURATION_SEC

FIGSIZE = (16, 9)
DPI = 120

BG = "#020510"
TEXT = "#c9d3df"
MUTED = "#6f8194"

CYAN = "#35c9ff"
BLUE = "#0b63ff"
ORANGE = "#ff9a3c"
MAGENTA = "#ff4dff"
GREEN = "#48ffb3"
RED = "#ff5a45"
WHITE = "#f0f6ff"

rng = np.random.default_rng(128)

# =========================
# Scene setup
# =========================

W, H = 16, 9

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_position([0, 0, 1, 1])

ax.set_xlim(0, W)
ax.set_ylim(0, H)
ax.set_aspect("equal")
ax.axis("off")

# =========================
# Helpers
# =========================

def smoothstep(edge0, edge1, value):
    t = np.clip((value - edge0) / (edge1 - edge0), 0, 1)
    return t * t * (3 - 2 * t)

def pulse(t, freq=6, phase=0):
    return 0.5 + 0.5 * np.sin(2 * np.pi * (freq * t + phase))

# =========================
# Title
# =========================

ax.text(
    0.65,
    8.55,
    "NEURAL NETWORK — LAYERED MODEL",
    color=TEXT,
    fontsize=20,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=50
)

stage_text = ax.text(
    0.65,
    8.20,
    "forward pass: activations move from input to output",
    color=MUTED,
    fontsize=12,
    fontweight="bold",
    ha="left",
    va="center",
    zorder=50
)

# =========================
# Network geometry
# =========================

layers = [
    ("INPUT", 5, 1.8, CYAN),
    ("HIDDEN 1", 7, 4.6, BLUE),
    ("HIDDEN 2", 6, 7.4, ORANGE),
    ("HIDDEN 3", 5, 10.2, MAGENTA),
    ("OUTPUT", 3, 13.0, GREEN),
]

node_positions = []
nodes = []
node_labels = []

for layer_idx, (name, count, x, color) in enumerate(layers):
    y_positions = np.linspace(2.0, 6.7, count)

    ax.text(
        x,
        7.25,
        name,
        color=color,
        fontsize=11,
        fontweight="bold",
        ha="center",
        va="center",
        zorder=40
    )

    layer_nodes = []

    for j, y in enumerate(y_positions):
        node = Circle(
            (x, y),
            0.16,
            facecolor="#07111f",
            edgecolor=color,
            linewidth=1.4,
            alpha=0.95,
            zorder=20
        )
        ax.add_patch(node)

        core = Circle(
            (x, y),
            0.055,
            facecolor=color,
            edgecolor="none",
            alpha=0.20,
            zorder=21
        )
        ax.add_patch(core)

        layer_nodes.append((node, core, x, y, color))

    nodes.append(layer_nodes)
    node_positions.append([(x, y) for _, _, x, y, _ in layer_nodes])

# =========================
# Edges
# =========================

edges = []

for l in range(len(node_positions) - 1):
    left = node_positions[l]
    right = node_positions[l + 1]

    for i, (x1, y1) in enumerate(left):
        for j, (x2, y2) in enumerate(right):
            # not fully connected visually, to avoid clutter
            if rng.random() < 0.72:
                weight = rng.uniform(0.4, 1.6)
                sign = rng.choice([-1, 1], p=[0.35, 0.65])
                color = CYAN if sign > 0 else RED

                line, = ax.plot(
                    [x1 + 0.16, x2 - 0.16],
                    [y1, y2],
                    color=color,
                    linewidth=0.35 * weight,
                    alpha=0.10,
                    zorder=5
                )

                edges.append({
                    "line": line,
                    "layer": l,
                    "weight": weight,
                    "sign": sign,
                    "phase": rng.uniform(0, 1),
                    "x1": x1 + 0.16,
                    "y1": y1,
                    "x2": x2 - 0.16,
                    "y2": y2,
                })

# =========================
# Moving activation particles
# =========================

particles = []
N_PARTICLES = 42

for i in range(N_PARTICLES):
    dot = ax.scatter(
        [],
        [],
        s=rng.uniform(15, 35),
        color=WHITE,
        alpha=0.0,
        zorder=30
    )

    particles.append({
        "artist": dot,
        "edge": rng.choice(edges),
        "delay": rng.uniform(0, 1),
        "speed": rng.uniform(0.9, 1.4),
        "color": rng.choice([CYAN, BLUE, ORANGE, MAGENTA, GREEN]),
    })

# =========================
# Loss / output panel
# =========================

loss_x = np.linspace(13.7, 15.3, 80)
loss_y = 2.3 + 0.9 * np.exp(-np.linspace(0, 3.2, 80))

loss_glow, = ax.plot([], [], color=RED, linewidth=7, alpha=0.0, zorder=25)
loss_line, = ax.plot([], [], color=RED, linewidth=2.0, alpha=0.0, zorder=26)

loss_label = ax.text(
    14.50,
    3.45,
    "loss",
    color=RED,
    fontsize=12,
    fontweight="bold",
    ha="center",
    va="center",
    alpha=0.0,
    zorder=30
)

target_label = ax.text(
    13.92,
    5.95,
    "target",
    color=TEXT,
    fontsize=10,
    ha="left",
    va="center",
    alpha=0.0,
    zorder=30
)

prediction_label = ax.text(
    13.92,
    5.55,
    "prediction",
    color=GREEN,
    fontsize=10,
    ha="left",
    va="center",
    alpha=0.0,
    zorder=30
)

# =========================
# Backprop arrows
# =========================

backprop_arrows = []

for i in range(len(layers) - 1, 0, -1):
    x1 = layers[i][2] - 0.42
    x2 = layers[i - 1][2] + 0.42
    y = 1.25

    arr = FancyArrowPatch(
        (x1, y),
        (x2, y),
        arrowstyle="->",
        mutation_scale=15,
        linewidth=1.3,
        color=RED,
        alpha=0.0,
        zorder=35
    )
    ax.add_patch(arr)
    backprop_arrows.append(arr)

backprop_label = ax.text(
    7.4,
    0.82,
    "backpropagation: error signal updates weights",
    color=RED,
    fontsize=11,
    fontweight="bold",
    ha="center",
    va="center",
    alpha=0.0,
    zorder=40
)

# =========================
# Animation update
# =========================

def update(frame):
    t = frame / (FRAMES - 1)
    global_a = smoothstep(0.02, 0.08, t) * (1.0 - smoothstep(0.94, 0.99, t))

    # phases
    forward_a = smoothstep(0.08, 0.46, t) * (1.0 - smoothstep(0.72, 0.86, t)) * global_a
    output_a = smoothstep(0.42, 0.62, t) * global_a
    loss_a = smoothstep(0.52, 0.72, t) * global_a
    back_a = smoothstep(0.66, 0.84, t) * global_a
    update_a = smoothstep(0.74, 0.92, t) * global_a

    # stage text
    if t < 0.40:
        stage_text.set_text("forward pass: activations move from input to output")
        stage_text.set_color(CYAN)
    elif t < 0.62:
        stage_text.set_text("prediction is compared with the target")
        stage_text.set_color(GREEN)
    elif t < 0.76:
        stage_text.set_text("loss measures the error")
        stage_text.set_color(RED)
    else:
        stage_text.set_text("backpropagation changes connection weights")
        stage_text.set_color(ORANGE)

    # edge activation / weight update
    for e in edges:
        layer_phase = e["layer"] / max(1, len(layers) - 2)
        f = smoothstep(layer_phase * 0.42, layer_phase * 0.42 + 0.22, t)

        base_alpha = 0.07 + 0.12 * f * forward_a
        upd = 0.20 * update_a * pulse(t, 5, e["phase"])

        e["line"].set_alpha((base_alpha + upd) * global_a)
        e["line"].set_linewidth(0.35 * e["weight"] + 0.75 * upd)

    # node activation wave
    for l_idx, layer_nodes in enumerate(nodes):
        layer_t = l_idx / max(1, len(nodes) - 1)
        wave = smoothstep(0.10 + layer_t * 0.35, 0.25 + layer_t * 0.35, t)

        for j, (node, core, x, y, color) in enumerate(layer_nodes):
            local_p = pulse(t, 5, j * 0.08 + l_idx * 0.11)
            node.set_alpha(0.85 * global_a)
            node.set_linewidth(1.2 + 1.6 * forward_a * wave * local_p)

            core.set_alpha((0.12 + 0.65 * forward_a * wave * local_p + 0.30 * update_a) * global_a)
            core.set_radius(0.055 + 0.055 * forward_a * wave * local_p + 0.035 * update_a)

    # moving particles along edges
    for p in particles:
        dot = p["artist"]
        e = p["edge"]

        u = ((t * p["speed"] + p["delay"]) % 1.0)

        # particles visible mostly during forward pass
        px = e["x1"] * (1 - u) + e["x2"] * u
        py = e["y1"] * (1 - u) + e["y2"] * u

        dot.set_offsets([[px, py]])
        dot.set_color(p["color"])
        dot.set_alpha(0.75 * forward_a * smoothstep(0.05, 0.20, u) * (1.0 - smoothstep(0.80, 0.98, u)))

    # loss curve
    n = int(1 + (len(loss_x) - 1) * smoothstep(0.52, 0.88, t))
    loss_line.set_data(loss_x[:n], loss_y[:n])
    loss_glow.set_data(loss_x[:n], loss_y[:n])
    loss_line.set_alpha(0.95 * loss_a)
    loss_glow.set_alpha(0.13 * loss_a)
    loss_label.set_alpha(0.95 * loss_a)

    # prediction labels
    target_label.set_alpha(0.85 * output_a)
    prediction_label.set_alpha(0.95 * output_a)

    # output nodes pulse
    for j, (node, core, x, y, color) in enumerate(nodes[-1]):
        core.set_alpha((0.20 + 0.75 * output_a * pulse(t, 5, j * 0.12)) * global_a)

    # backprop arrows
    for i, arr in enumerate(backprop_arrows):
        local = smoothstep(0.66 + i * 0.035, 0.76 + i * 0.035, t)
        arr.set_alpha(0.75 * back_a * local)

    backprop_label.set_alpha(0.95 * back_a)

    return (
        [stage_text, loss_line, loss_glow, loss_label, target_label, prediction_label, backprop_label]
        + [e["line"] for e in edges]
        + [artist for layer in nodes for node_pair in layer for artist in node_pair[:2]]
        + [p["artist"] for p in particles]
        + backprop_arrows
    )

# =========================
# Save GIF
# =========================

anim = FuncAnimation(
    fig,
    update,
    frames=FRAMES,
    interval=1000 / FPS,
    blit=False
)

writer = PillowWriter(fps=FPS)
anim.save(GIF_PATH, writer=writer)

plt.close(fig)

display(Image(filename=str(GIF_PATH)))
print(f"Saved: {GIF_PATH}")